# Complete LLM Zoomcamp Module 2 Notes: Vector Search & Embeddings with Google Gemini

---

## Part 1: Vector Search Fundamentals

### 01 - Introduction to Vector Search

#### Limitations of Keyword Search

Keyword search engines (like the basic implementations of `minsearch` and `sqlitesearch`) rely strictly on exact word matching. If a document lacks the specific term searched for, it will not be retrieved, even if the semantic meaning is identical.

Consider these two participant queries:

* *"Can I still join the course after the start date?"*

* *"Is it possible to enroll late?"*


While they share almost no identical vocabulary, their core intent is identical. Lexical search engines fail here, necessitating an architecture that matches conceptual ideas rather than literal characters—**Vector Search**.

#### The Vector Search Lifecycle

Vector search executes across two distinct processing phases:

```mermaid
flowchart TD
    subgraph Offline ["Offline Phase (Indexing)"]
        Docs[Raw Documents] --> EmbedModel1[Embedding Model]
        EmbedModel1 --> Vectors[Dense Arrays / Embeddings]
        Vectors --> VDB[(Vector Index)]
    end
    subgraph Online ["Online Phase (Querying)"]
        Query[User Query] --> EmbedModel2[Embedding Model]
        EmbedModel2 --> QueryVec[Query Vector]
        QueryVec --> Similarity[Similarity Comparison]
        VDB --> Similarity
        Similarity --> Results[Top N Nearest Neighbors]
    end

```

1. **Offline Phase (Indexing):** Text documents are passed through a neural network model to convert them into dense arrays of numbers (embeddings) and stored inside a specialized index.


2. **Online Phase (Querying):** The incoming user query is embedded via the exact same model. The system then performs a geometric proximity search to retrieve document vectors closest to the query vector.



#### Mathematical Distance Metrics

To evaluate how close two concepts land in the vector space, we calculate their proximity using a metric like **Cosine Similarity**. Cosine similarity measures the geometric angle between two non-zero vectors:

$$\text{Cosine Similarity} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$

* **$\cos(\theta) \approx 1$ (Angle close to $0^\circ$):** The vectors point in virtually identical directions, indicating highly similar text semantics.


* **$\cos(\theta) \approx 0$ (Angle close to $90^\circ$):** The vectors are orthogonal, meaning the texts are conceptually unrelated.


* **$\cos(\theta) \approx -1$ (Angle close to $180^\circ$):** The vectors point in opposite directions, reflecting diametrically opposed meanings.



> **Note:** In practical text embeddings, scores rarely drop below $0$ because the models map spatial properties primarily into regions with positive components.
> 
> 

---

### 02 - Text Embeddings & Local Setup

#### Word vs Sentence Space

While early implementations like `word2vec` embedded individual words in isolation, modern dense retrieval uses sentence-level encoders. These models dynamically evaluate spatial coordinates based on context. For instance, the token *"judge"* maps to distinctly different vector positions depending on whether it is used in a legal framework or a machine learning context (*"LLM-as-a-judge"*).

#### Environment Configuration

Because open-source embedding frameworks pull heavily down onto PyTorch and localized dependencies, configuring a separate runtime environment is highly recommended.

Initialize your project workspace using the `uv` package manager:

```bash
uv init
uv add requests minsearch google-generativeai jupyter python-dotenv sentence-transformers tqdm numpy
```

#### Selecting an Embedding Model

We utilize the open-source `all-MiniLM-L6-v2` model from the `sentence-transformers` library. It features compact representations, runs efficiently on a standard CPU, and performs excellently on short English phrases:

* **Vector Dimensions:** $384$ fields per array.


* **Distance Metric:** Presets optimal performance using dot-product calculations on pre-normalized vectors.





In [1]:
from sentence_transformers import SentenceTransformer

# Automatically handles local caching after the first Hugging Face fetch (~80 MB)
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [4]:
v1.dot(dv)

np.float32(0.32332402)

In [5]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [6]:
v2.dot(dv)

np.float32(0.019730477)

In [7]:
import numpy as np

# Encode two distinct queries and a benchmark document
q1 = model.encode("Can I still join the course after the start date?")
q2 = model.encode("How to install Docker on Windows?")
doc = model.encode("You don't need to register. You're accepted. Start learning and submitting homework.")

# Check vector sizes
print(f"Vector dimensions: {q1.shape}") # Output: (384,)

# Execute geometric evaluations via Numpy dot product
print(f"Semantic match similarity: {q1.dot(doc):.4f}")    # Higher score (~0.32)
print(f"Unrelated phrase similarity: {q2.dot(doc):.4f}")   # Near-zero score (~0.01)

Vector dimensions: (384,)
Semantic match similarity: 0.2646
Unrelated phrase similarity: 0.0139


In [8]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

'wget' is not recognized as an internal or external command,
operable program or batch file.


# 1.3 Embedding Our Dataset

In [9]:
%%writefile ingest.py
import requests

def load_faq_data():
    docs_url = "https://datatalks.club/faq/json/courses.json"
    response = requests.get(docs_url)
    courses_raw = response.json()
    
    documents = []
    url_prefix = "https://datatalks.club/faq"
    
    for course in courses_raw:
        course_url = f"{url_prefix}{course['path']}"
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()
        documents.extend(course_data)
    
    return documents

Overwriting ingest.py


In [10]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Successfully loaded {len(documents)} documents!")

Successfully loaded 1350 documents!


In [11]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [12]:
from tqdm.auto import tqdm

In [13]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [14]:
import numpy as np
X = np.array(vectors)

In [15]:
X.shape

(1350, 384)

# 1.4 Vector Search

In [16]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [17]:
scores = X.dot(v_query)

In [18]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [19]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294094))

In [20]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [21]:
top5 = np.argsort(scores)[-5:]

In [22]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [23]:
import requests

def load_faq_data():
    docs_url = "https://datatalks.club/faq/json/courses.json"
    response = requests.get(docs_url)
    courses_raw = response.json()
    
    documents = []
    url_prefix = "https://datatalks.club/faq"
    
    for course in courses_raw:
        course_url = f"{url_prefix}{course['path']}"
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()
        documents.extend(course_data)
    
    return documents

# Load the documents directly
documents = load_faq_data()
print(f"Successfully loaded {len(documents)} documents!")

Successfully loaded 1350 documents!


In [24]:
import sys
import os

# Option A: If ingest.py is in the parent folder
sys.path.append('..')

# Option B: If ingest.py is inside a folder named 'code' in the parent directory
# sys.path.append('../code')

from ingest import load_faq_data
documents = load_faq_data()
print(f"Successfully loaded {len(documents)} documents!")

Successfully loaded 1350 documents!


In [25]:
import requests

# We define the function directly here instead of importing it
def load_faq_data():
    docs_url = "https://datatalks.club/faq/json/courses.json"
    response = requests.get(docs_url)
    courses_raw = response.json()
    
    documents = []
    url_prefix = "https://datatalks.club/faq"
    
    for course in courses_raw:
        course_url = f"{url_prefix}{course['path']}"
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()
        documents.extend(course_data)
    
    return documents

# Execute the function and load the documents into memory
documents = load_faq_data()
print(f"Success! Loaded {len(documents)} documents.")

Success! Loaded 1350 documents.


In [26]:
top5 = np.argsort(scores)[-5:]

In [27]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [28]:
scores[top5]

TypeError: only integer scalar arrays can be converted to a scalar index

In [29]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294094
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Relate

In [30]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [31]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [32]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [33]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [34]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [35]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [38]:
import requests
from minsearch import Index

# 1. Define the data loading logic directly in the notebook
def load_faq_data():
    docs_url = "https://datatalks.club/faq/json/courses.json"
    response = requests.get(docs_url)
    courses_raw = response.json()
    
    documents = []
    url_prefix = "https://datatalks.club/faq"
    
    for course in courses_raw:
        course_url = f"{url_prefix}{course['path']}"
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()
        documents.extend(course_data)
    
    return documents

# 2. Define the index building logic directly in the notebook
def build_index(documents):
    index = Index(
        text_fields=["question", "section", "answer"],
        keyword_fields=["course"]
    )
    index.fit(documents)
    return index

# 3. Execute both functions (No 'ingest' import required!)
documents = load_faq_data()
index = build_index(documents)

print(f"Success! Indexed {len(documents)} documents into minsearch.")

Success! Indexed 1350 documents into minsearch.


In [40]:
# Let's test the index we just successfully built!
query = "I just discovered the course. Can I still join it?"

# We can query the 'index' variable directly since it is already in memory
search_results = index.search(
    query,
    num_results=5,
    boost_dict={"question": 3.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"}
)

# Print the top result to prove it works
print("Top Result Match:")
print(search_results[0]['question'])
print(search_results[0]['answer'])

Top Result Match:
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [41]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [42]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [43]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [44]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [53]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

C:\Users\user\AppData\Local\Temp\ipykernel_25088\4020778107.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [55]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

if api_key:
    print(f"Key loaded successfully! Starts with: {api_key[:5]}...")
else:
    print("ERROR: API key is missing or None. Python cannot find your .env file.")

ERROR: API key is missing or None. Python cannot find your .env file.


In [56]:
import google.generativeai as genai

# Hardcode the key directly just to get past this blocker
# (Make sure to delete the key from the notebook before saving or pushing to GitHub!)
genai.configure(api_key="YOUR_ACTUAL_API_KEY_HERE")

In [58]:
# 1. Define the base RAG class directly in the notebook (Configured for Gemini)
class RAGBase:
    def __init__(self, index, course="llm-zoomcamp", model="gemini-3.1-flash-lite"):
        self.index = index
        self.course = course
        self.model = model
        self.instructions = """
        Your task is to answer questions from the course participants based on the provided context.
        Use the context to find relevant information and provide accurate answers. 
        If the answer is not found in the context, respond with "I don't know."
        """.strip()
        self.prompt_template = "QUESTION: {question}\n\nCONTEXT:\n{context}"
    
    def search(self, query, num_results=5):
        boost_dict = {"question": 3.0, "section": 0.5}
        filter_dict = {"course": self.course}
        return self.index.search(query, num_results=num_results, boost_dict=boost_dict, filter_dict=filter_dict)
    
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(doc["section"])
            lines.append("Q: " + doc["question"])
            lines.append("A: " + doc["answer"])
            lines.append("")
        return "\n".join(lines).strip()
    
    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(question=query, context=context)
    
    def llm(self, prompt):
        full_prompt = f"{self.instructions}\n\n{prompt}"
        model_instance = genai.GenerativeModel(self.model)
        response = model_instance.generate_content(full_prompt)
        return response.text
    
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer


# 2. Define the Vector Search subclass
class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        # Convert the incoming text query into a vector
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        # Search the vector index instead of the text index
        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

# 3. Initialize the Vector Assistant
# (This assumes 'model' is your SentenceTransformer and 'vindex' is your minsearch VectorSearch index from earlier steps)
assistant = RAGVector(
    embedder=model, 
    index=vindex
)

# 4. Test the pipeline
query = "I just found out about the program, can I still sign up?"
answer = assistant.rag(query)

print("Gemini's Answer:")
print(answer)

InvalidArgument: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]

In [51]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

NameError: name 'assistant' is not defined

In [ ]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

NameError: name 'RAGBase' is not defined